# Classifier Patch Extraction — Visual QA

This notebook visualises the standalone classifier's patch extraction pipeline:
1. Full preprocessing (no letterbox resize)
2. GT box → context expansion → integer-friendly crop snapping
3. Integer-snapped vs naive resize comparison
4. Augmented patch variants
5. Label distribution (pathology vs BI-RADS)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

from config import PreprocessConfig, Representation
from preprocess import MammogramPreprocessor
from dataset import build_sample_index
from classify_dataset import extract_patch, _snap_to_multiple, _augment_patch
import random

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

In [ ]:
# Load sample index (both splits)
DATA_ROOT = "../data/cbis-ddsm"
TRAIN_CSV = "../meta/cbis-ddsm/mass_case_description_train_set.csv"
TEST_CSV  = "../meta/cbis-ddsm/mass_case_description_test_set.csv"

train_samples = build_sample_index(os.path.join(DATA_ROOT, "train"), TRAIN_CSV)
test_samples  = build_sample_index(os.path.join(DATA_ROOT, "test"), TEST_CSV)

print(f"Train: {len(train_samples)} images, "
      f"{sum(len(s['masks']) for s in train_samples)} lesions")
print(f"Test:  {len(test_samples)} images, "
      f"{sum(len(s['masks']) for s in test_samples)} lesions")

## 1. Full preprocessing (no letterbox resize)

Show what the image looks like after orient → pectoral removal (off) → tight crop → denoise → MMS pseudo-color, but WITHOUT the final letterbox resize.

In [ ]:
# Preprocessor without letterbox resize
preprocessor = MammogramPreprocessor(PreprocessConfig(
    target_size=None,
    representation=Representation.MMS_PSEUDO_COLOR,
    remove_pectoral=False,
))

# Pick a few diverse samples
rng = random.Random(42)
demo_indices = rng.sample(range(len(train_samples)), min(6, len(train_samples)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, idx in zip(axes.flat, demo_indices):
    sample = train_samples[idx]
    image = cv2.imread(sample["image_path"], cv2.IMREAD_GRAYSCALE)
    image_3ch, tx = preprocessor(image, view=sample["view"])

    # Show the 3-channel image (RGB composite)
    display = np.clip(image_3ch, 0, 1)
    ax.imshow(display)
    ax.set_title(
        f"{sample['patient_id']} {sample['side']} {sample['view']}\n"
        f"density={sample['density']}  shape={image_3ch.shape[:2]}",
        fontsize=9,
    )
    ax.axis("off")

fig.suptitle("Preprocessing without letterbox resize (MMS pseudo-color)", fontsize=13)
fig.tight_layout()
plt.show()

## 2. GT box → context expansion → integer crop snapping

For each lesion, show:
- The GT bounding box (red)
- The expanded context region (yellow)
- The integer-snapped crop region (green)
- The final 224×224 patch

In [ ]:
PATCH_SIZE = 224
CONTEXT_FACTOR = 1.5

fig, axes = plt.subplots(4, 4, figsize=(18, 18))

sample_lesions = []
for idx in demo_indices[:4]:
    sample = train_samples[idx]
    if sample["masks"]:
        sample_lesions.append((sample, sample["masks"][0]))

for row, (sample, mask_info) in enumerate(sample_lesions):
    image = cv2.imread(sample["image_path"], cv2.IMREAD_GRAYSCALE)
    image_3ch, tx = preprocessor(image, view=sample["view"])
    h, w = image_3ch.shape[:2]

    raw_mask = cv2.imread(mask_info["path"], cv2.IMREAD_GRAYSCALE)
    raw_mask = (raw_mask > 127).astype(np.uint8) * 255
    transformed_mask = preprocessor.transform_mask(raw_mask, tx)

    binary = (transformed_mask > 127).astype(np.uint8)
    coords = cv2.findNonZero(binary)
    bx, by, bw, bh = cv2.boundingRect(coords)
    cx, cy = bx + bw // 2, by + bh // 2
    box_size = max(bw, bh)
    expanded = int(box_size * CONTEXT_FACTOR)
    crop_size = _snap_to_multiple(expanded, PATCH_SIZE)

    # Col 0: full image with boxes
    ax = axes[row, 0]
    ax.imshow(np.clip(image_3ch, 0, 1))
    ax.add_patch(mpatches.Rectangle(
        (bx, by), bw, bh, linewidth=2, edgecolor="red", facecolor="none", label="GT box"))
    half_exp = expanded // 2
    ax.add_patch(mpatches.Rectangle(
        (cx - half_exp, cy - half_exp), expanded, expanded,
        linewidth=2, edgecolor="yellow", facecolor="none", linestyle="--", label="Context"))
    half_crop = crop_size // 2
    x1 = max(0, cx - half_crop)
    y1 = max(0, cy - half_crop)
    ax.add_patch(mpatches.Rectangle(
        (x1, y1), crop_size, crop_size,
        linewidth=2, edgecolor="lime", facecolor="none", label="Snapped crop"))
    ax.set_title(f"{sample['patient_id']} {sample['view']}\n{w}×{h}", fontsize=9)
    ax.axis("off")
    if row == 0:
        ax.legend(fontsize=7, loc="lower right")

    # Col 1: cropped region at original resolution
    ax = axes[row, 1]
    x2 = min(w, x1 + crop_size)
    y2 = min(h, y1 + crop_size)
    crop_orig = image_3ch[y1:y2, x1:x2]
    ax.imshow(np.clip(crop_orig, 0, 1))
    ax.set_title(f"Crop: {crop_orig.shape[1]}×{crop_orig.shape[0]}", fontsize=9)
    ax.axis("off")

    # Col 2: integer-snapped resize to 224
    ax = axes[row, 2]
    patch_int, _ = extract_patch(
        image_3ch, transformed_mask, PATCH_SIZE, CONTEXT_FACTOR)
    ax.imshow(np.clip(patch_int, 0, 1))
    scale = crop_size // PATCH_SIZE
    ax.set_title(f"Integer resize ({scale}× down)\n{PATCH_SIZE}×{PATCH_SIZE}", fontsize=9)
    ax.axis("off")

    # Col 3: naive (non-integer) resize for comparison
    ax = axes[row, 3]
    naive_size = expanded  # not snapped
    nx1 = max(0, cx - naive_size // 2)
    ny1 = max(0, cy - naive_size // 2)
    nx2 = min(w, nx1 + naive_size)
    ny2 = min(h, ny1 + naive_size)
    naive_crop = image_3ch[ny1:ny2, nx1:nx2]
    naive_resized = cv2.resize(naive_crop, (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_AREA)
    ax.imshow(np.clip(naive_resized, 0, 1))
    naive_scale = naive_size / PATCH_SIZE
    ax.set_title(f"Naive resize ({naive_scale:.2f}× down)\n{PATCH_SIZE}×{PATCH_SIZE}", fontsize=9)
    ax.axis("off")

fig.suptitle(
    "Patch extraction: GT box (red) → context (yellow) → integer-snapped crop (green) → 224×224",
    fontsize=13,
)
fig.tight_layout()
plt.show()

## 3. Augmented patch variants

Show the same lesion patch with different augmentation rolls: offset, flipped, rotated, contrast jittered.

In [ ]:
# Pick one sample and show 8 augmented variants
sample = train_samples[demo_indices[0]]
mask_info = sample["masks"][0]

image = cv2.imread(sample["image_path"], cv2.IMREAD_GRAYSCALE)
image_3ch, tx = preprocessor(image, view=sample["view"])
raw_mask = cv2.imread(mask_info["path"], cv2.IMREAD_GRAYSCALE)
raw_mask = (raw_mask > 127).astype(np.uint8) * 255
transformed_mask = preprocessor.transform_mask(raw_mask, tx)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

# First: no augmentation
patch_clean, _ = extract_patch(image_3ch, transformed_mask, PATCH_SIZE, CONTEXT_FACTOR)
axes[0, 0].imshow(np.clip(patch_clean, 0, 1))
axes[0, 0].set_title("No augmentation", fontsize=9)
axes[0, 0].axis("off")

# Then: 9 different augmentation rolls
# Rotation is now applied pre-crop inside extract_patch (no black corners)
for i, ax in enumerate(axes.flat[1:]):
    aug_rng = random.Random(i * 7 + 1)
    patch, _ = extract_patch(
        image_3ch, transformed_mask, PATCH_SIZE, CONTEXT_FACTOR,
        offset_jitter=0.2, rotation_degrees=15.0, rng=aug_rng,
    )
    patch = _augment_patch(patch, aug_rng)
    ax.imshow(np.clip(patch, 0, 1))
    ax.set_title(f"Augmented #{i + 1}", fontsize=9)
    ax.axis("off")

fig.suptitle(
    f"Augmentation variants — {sample['patient_id']} {sample['view']} "
    f"({mask_info['pathology_label']})",
    fontsize=13,
)
fig.tight_layout()
plt.show()

## 4. Label distribution (pathology vs BI-RADS)

In [ ]:
# Count labels across both label sources
path_labels = []
assess_labels = []
assess_raw = []

for sample in train_samples:
    for mask in sample["masks"]:
        path_labels.append(mask["pathology_label"])
        assess_labels.append(mask["assessment_label"])
        assess_raw.append(mask["assessment"])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Pathology labels
path_counts = Counter(path_labels)
ax = axes[0]
bars = ax.bar(path_counts.keys(), path_counts.values(), color=["#4CAF50", "#F44336", "#9E9E9E"])
ax.set_title("Pathology labels (biopsy)")
ax.set_ylabel("Count")
for bar, count in zip(bars, path_counts.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(count), ha="center", fontsize=10)

# Assessment-derived labels
assess_counts = Counter(assess_labels)
ax = axes[1]
bars = ax.bar(assess_counts.keys(), assess_counts.values(), color=["#4CAF50", "#F44336", "#9E9E9E"])
ax.set_title("BI-RADS assessment labels\n(0-3→BENIGN, 4-5→MALIGNANT)")
ax.set_ylabel("Count")
for bar, count in zip(bars, assess_counts.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(count), ha="center", fontsize=10)

# Raw assessment score distribution
assess_raw_counts = Counter(assess_raw)
ax = axes[2]
scores = sorted(assess_raw_counts.keys())
ax.bar([str(s) for s in scores], [assess_raw_counts[s] for s in scores])
ax.set_title("Raw BI-RADS assessment scores")
ax.set_xlabel("Assessment score")
ax.set_ylabel("Count")

fig.suptitle("Training set label distributions", fontsize=13)
fig.tight_layout()
plt.show()

# Cross-tabulation
print("\nCross-tabulation: pathology vs assessment-derived label")
print(f"{'':>15} | {'path=BENIGN':>12} | {'path=MALIGNANT':>14} | {'path=UNKNOWN':>12}")
print("-" * 65)
for al in ["BENIGN", "MALIGNANT", "UNKNOWN"]:
    row = []
    for pl in ["BENIGN", "MALIGNANT", "UNKNOWN"]:
        count = sum(1 for p, a in zip(path_labels, assess_labels) if p == pl and a == al)
        row.append(count)
    print(f"assess={al:>8} | {row[0]:>12} | {row[1]:>14} | {row[2]:>12}")